# Coleta de dados do Bitcoin

Este notebook usa `yfinance` para baixar cinco anos de dados diários de Bitcoin (`BTC-USD`) e salvar em `dados/bitcoin_5_anos.csv`. Os preços são em dólares americanos.

Fonte: [Yahoo Finance — BTC-USD](https://finance.yahoo.com/quote/BTC-USD/history/). Execute as células de cima para baixo, com conexão à internet.

## 1. Instalar as bibliotecas

O comando `%pip` instala as bibliotecas no ambiente Python usado pelo notebook.

In [1]:
%pip install -q yfinance pandas


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /home/kaian/Documentos/Murilao/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import yfinance as yf

print(f'Versão do yfinance: {yf.__version__}')

Versão do yfinance: 1.7.0


## 2. Definir a moeda e o período

A data final é o dia atual em UTC. No `yfinance`, `end` é exclusivo: o dia atual fica fora da coleta, evitando usar um fechamento ainda em andamento. A data inicial é cinco anos antes da data final.

Referência: [documentação de yfinance.download](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html).

In [ ]:
moeda = 'BTC-USD'
anos = 5
data_fim = datetime.now(timezone.utc).date()
data_inicio = (pd.Timestamp(data_fim) - pd.DateOffset(years=anos)).date()
arquivo_csv = Path('dados/bitcoin_5_anos.csv')

print(f'Moeda: {moeda}')
print(f'Data inicial: {data_inicio}')
print(f'Data final exclusiva: {data_fim}')

## 3. Baixar e organizar os dados

`interval='1d'` solicita uma linha por dia. `auto_adjust=False` mantém os preços sem ajuste automático. `multi_level_index=False` deixa o cabeçalho simples para exportar em CSV.

As colunas são `Date` (data), `Open` (abertura), `High` (máxima), `Low` (mínima), `Close` (fechamento) e `Volume` (volume informado pelo Yahoo Finance).

In [4]:
dados = yf.download(
    tickers=moeda,
    start=data_inicio.isoformat(),
    end=data_fim.isoformat(),
    interval='1d',
    auto_adjust=False,
    multi_level_index=False,
    progress=False,
    threads=False,
    timeout=30,
)

if dados is None or dados.empty:
    raise RuntimeError('Nenhum dado recebido. Confira a conexão e tente executar esta célula novamente.')

# Selecionar as colunas e ordenar da data mais antiga para a mais recente.
dados = dados[['Open', 'High', 'Low', 'Close', 'Volume']].sort_index()
# Transformar o índice de datas em uma coluna do CSV.
dados = dados.rename_axis('Date').reset_index()

dados.head()

,Date,Open,High,Low,Close,Volume
0,2021-10-05,49174.960938,51839.984375,49072.839844,51514.812500,35873904236
1,2021-10-06,51486.664062,55568.464844,50488.191406,55361.449219,49034730168
2,2021-10-07,55338.625000,55338.625000,53525.468750,53805.984375,36807860413
3,2021-10-08,53802.144531,55922.980469,53688.054688,53967.847656,34800873924
4,2021-10-09,53929.781250,55397.945312,53735.144531,54968.222656,32491211414


## 4. Salvar e conferir o CSV

O arquivo é salvo na pasta `dados/`, criada automaticamente se necessário. `index=False` evita uma coluna extra com a numeração das linhas. Executar novamente substitui o CSV com a nova coleta.

Depois, o arquivo é lido para conferir a quantidade de registros, as datas e os valores ausentes. Os dados não são preenchidos nem embaralhados.

In [ ]:
arquivo_csv.parent.mkdir(parents=True, exist_ok=True)
dados.to_csv(arquivo_csv, index=False, date_format='%Y-%m-%d', encoding='utf-8')
print(f'CSV salvo em: {arquivo_csv.resolve()}')

In [6]:
dados_csv = pd.read_csv(arquivo_csv, parse_dates=['Date'])

print(f'Total de registros: {len(dados_csv)}')
print(f'Primeira data: {dados_csv["Date"].min().date()}')
print(f'Última data: {dados_csv["Date"].max().date()}')
print(f'Datas duplicadas: {dados_csv["Date"].duplicated().sum()}')
print('Valores ausentes por coluna:')
print(dados_csv.isna().sum().to_string())

dados_csv.tail()

Total de registros: 1826
Primeira data: 2021-10-05
Última data: 2026-10-04
Datas duplicadas: 0
Valores ausentes por coluna:
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0


,Date,Open,High,Low,Close,Volume
1821,2026-09-30,83621.398438,85599.804688,82927.875000,83553.851562,36290317874
1822,2026-10-01,83554.250000,85224.273438,83132.757812,84853.101562,32802609098
1823,2026-10-02,84855.648438,87146.351562,83852.695312,84497.210938,46528418756
1824,2026-10-03,84503.187500,85019.328125,84429.648438,84763.578125,12574826191
1825,2026-10-04,84765.515625,86766.148438,84707.320312,86480.304688,15031650242
